In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [3]:
!wget -q https://repo1.maven.org/maven2/com/mysql/mysql-connector-j/8.0.33/mysql-connector-j-8.0.33.jar -O /tmp/mysql-connector-java-8.0.33.jar

In [4]:
spark = (
    SparkSession
    .builder
    .appName("Stream From Kafka")
    .config("spark.jars", "/tmp/mysql-connector-java-8.0.33.jar")
    .config("spark.driver.extraClassPath", "/tmp/mysql-connector-java-8.0.33.jar") 
    .config("spark.executor.extraClassPath", "/tmp/mysql-connector-java-8.0.33.jar") 
    .config("spark.streaming.stopGracefullyOnShutdown","true")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0")
    .config("spark.sql.shuffle.partitions",8)
    .master("local[*]")
    .getOrCreate()
)
spark

In [5]:
kafka_df = (
    spark
    .readStream
    .format("kafka")
    .option("kafka.bootstrap.servers","ed-kafka-1:29092,ed-kafka-2:29092,ed-kafka-3:29092")
    .option("subscribe","finance")
    .option("startingOffsets","earliest")
    .load()
)
kafka_df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [6]:
#kafka_df.show()

In [7]:
kafka_df = kafka_df.withColumns({
    "key":expr("cast(key as string)"),
    "value":expr("cast(value as string)")
})
#kafka_df.show()

In [8]:
from pyspark.sql.types import StructType , StructField , StringType , IntegerType , FloatType , TimestampType

json_schema = StructType([
    StructField("symbol",StringType(),nullable=True),
    StructField("timestamp",TimestampType(),nullable=True),
    StructField("open",FloatType(),nullable=True),
    StructField("high",FloatType(),nullable=True),
    StructField("low",FloatType(),nullable=True),
    StructField("close",FloatType(),nullable=True),
    StructField("volume",IntegerType(),nullable=True),
    StructField("company_name",StringType(),nullable=True),
    StructField("sector",StringType(),nullable=True),
    StructField("industry",StringType(),nullable=True),
    StructField("country",StringType(),nullable=True),
    StructField("currency",StringType(),nullable=True),
    StructField("market_cap",StringType(),nullable=True),
    StructField("zip",StringType(),nullable=True),
    StructField("website",StringType(),nullable=True)
])

In [9]:
json_kafka = kafka_df.withColumn("json_value",from_json("value",json_schema)).select("json_Value.*")
#json_kafka.show()

In [10]:
json_kafka = json_kafka.withColumn("company_name",regexp_replace(split("company_name"," ").getItem(0),",",""))
#json_kafka.show()

In [11]:
json_kafka = json_kafka.withColumns({
    "company_name":regexp_replace("company_name",".com",""),
    "website":ltrim("website"),
    "zip":split("zip","-").getItem(0)
}
)

In [11]:
#json_kafka.show()

+------+-------------------+-------+------+--------+------+-------+------------+--------------------+--------------------+-------------+--------+-------------+-----+--------------------+
|symbol|          timestamp|   open|  high|     low| close| volume|company_name|              sector|            industry|      country|currency|   market_cap|  zip|             website|
+------+-------------------+-------+------+--------+------+-------+------------+--------------------+--------------------+-------------+--------+-------------+-----+--------------------+
|  AAPL|2025-12-31 20:59:00|272.045|272.13|  271.75|271.84| 729227|       Apple|          Technology|Consumer Electronics|United States|     USD|4034508095488|95014|https://www.apple...|
| GOOGL|2025-12-31 20:59:00| 313.37|313.43|  312.96| 313.0| 336596|    Alphabet|Communication Ser...|Internet Content ...|United States|     USD|3791094022144|94043|     https://abc.xyz|
|  MSFT|2025-12-31 20:59:00| 483.77|483.88|  483.38|483.65| 35430

In [12]:
json_kafka = json_kafka \
    .withColumnRenamed("sector", "sector_name") \
    .withColumnRenamed("industry", "name_ind") \
    .withColumnRenamed("country", "country_name") 

In [13]:
import logging
def save_to_mysql(df,batch_id):
    # Save Sector:
    logging.info("Savring Sectors")
    # First, read existing sectors from MySQL
    try:
        existing_data = spark.read \
            .format("jdbc") \
            .option("url", "jdbc:mysql://host.docker.internal:3306/kafka_spark") \
            .option("dbtable", "sector") \
            .option("user", "root") \
            .option("password", "abdellah2004") \
            .option("driver", "com.mysql.cj.jdbc.Driver") \
            .load()
        
        # Get list of existing sector names
        existing_data = [row.sector_name for row in existing_data.collect()]
        
        # Get new sectors from Kafka data
        new_data = df.select("sector_name").distinct()
        
        # Filter out sectors that already exist
        data_to_insert = new_data.filter(~col("sector_name").isin(existing_data))
        
        # Insert only new sectors
        if data_to_insert.count() > 0:
            (
                data_to_insert.write
                .format("jdbc")
                .option("url", "jdbc:mysql://host.docker.internal:3306/kafka_spark") 
                .option("dbtable", "sector")
                .option("user", "root")
                .option("password", "abdellah2004")
                .option("driver", "com.mysql.cj.jdbc.Driver")
                .mode("append")
                .save()
            )
            logging.ingo(f"Inserted {data_to_insert.count()} new sectors")
    except Exception as e:
        logging.error(f"ERROR - {str(e)}")

    try:
        logging.info("Saving Industry ...")
        existing_data = spark.read \
            .format("jdbc") \
            .option("url", "jdbc:mysql://host.docker.internal:3306/kafka_spark") \
            .option("dbtable", "industry") \
            .option("user", "root") \
            .option("password", "abdellah2004") \
            .option("driver", "com.mysql.cj.jdbc.Driver") \
            .load()
        existing_data = [row.name_ind for row in existing_data.collect()]
        new_data = df.select("name_ind").distinct()
        data_to_insert = new_data.filter(~col("name_ind").isin(existing_data))
        if data_to_insert.count() > 0:
            (
                data_to_insert.write
                .format("jdbc")
                .option("url", "jdbc:mysql://host.docker.internal:3306/kafka_spark") 
                .option("dbtable", "industry")
                .option("user", "root")
                .option("password", "abdellah2004")
                .option("driver", "com.mysql.cj.jdbc.Driver")
                .mode("append")
                .save()
            )
            logging.info(f"Inserted {data_to_insert.count()} new industry")
    except Exception as e:
        logging.error(f"ERROR - {str(e)}")

    try:
        logging.info("Saving Countries ...")
        existing_data = spark.read \
        .format("jdbc") \
        .option("url", "jdbc:mysql://host.docker.internal:3306/kafka_spark") \
        .option("dbtable", "country") \
        .option("user", "root") \
        .option("password", "abdellah2004") \
        .option("driver", "com.mysql.cj.jdbc.Driver") \
        .load()
        existing_data = [row.country_name for row in existing_data.collect()]
        new_data = df.select("country_name").distinct()
        data_to_insert = new_data.filter(~col("country_name").isin(existing_data))
        if data_to_insert.count() > 0:
            (
                data_to_insert.write
                .format("jdbc")
                .option("url", "jdbc:mysql://host.docker.internal:3306/kafka_spark") 
                .option("dbtable", "country")
                .option("user", "root")
                .option("password", "abdellah2004")
                .option("driver", "com.mysql.cj.jdbc.Driver")
                .mode("append")
                .save()
            )
            logging.info(f"Inserted {data_to_insert.count()} new country")
    except Exception as e:
        logging.error(f"ERROR - {str(e)}")

    ### Saving Company Data
    try:
        logging.info("Saving Company ...")
        sectors = spark.read \
            .format("jdbc") \
            .option("url", "jdbc:mysql://host.docker.internal:3306/kafka_spark") \
            .option("dbtable", "sector") \
            .option("user", "root") \
            .option("password", "abdellah2004") \
            .option("driver", "com.mysql.cj.jdbc.Driver") \
            .load()

        countries = spark.read \
            .format("jdbc") \
            .option("url", "jdbc:mysql://host.docker.internal:3306/kafka_spark") \
            .option("dbtable", "country") \
            .option("user", "root") \
            .option("password", "abdellah2004") \
            .option("driver", "com.mysql.cj.jdbc.Driver") \
            .load()

        industries = spark.read \
            .format("jdbc") \
            .option("url","jdbc:mysql://host.docker.internal:3306/kafka_spark") \
            .option("dbtable","industry") \
            .option("user","root") \
            .option("password","abdellah2004") \
            .option("driver", "com.mysql.cj.jdbc.Driver") \
            .load()

        
        company = df.join(sectors,on="sector_name",how="left") \
            .join(countries,on="country_name",how="left") \
            .join(industries,on="name_ind",how="left") \
            .select("symbol","company_name","website","id_sector","id_ind","id_country","currency","zip")

        existing_data = spark.read \
            .format("jdbc") \
            .option("url", "jdbc:mysql://host.docker.internal:3306/kafka_spark") \
            .option("dbtable", "company") \
            .option("user", "root") \
            .option("password", "abdellah2004") \
            .option("driver", "com.mysql.cj.jdbc.Driver") \
            .load()
        existing_data = [row.symbol for row in existing_data.collect()]
        new_data = company.select("symbol").distinct()
        data_to_insert = new_data.filter(~col("symbol").isin(existing_data))
        if data_to_insert.count() > 0:
            try: 
                logging.info("Saving to Finance Data")
                (
                    data_to_insert.write
                    .format("jdbc")
                    .option("url", "jdbc:mysql://host.docker.internal:3306/kafka_spark") 
                    .option("dbtable", "company")
                    .option("user", "root")
                    .option("password", "abdellah2004")
                    .option("driver", "com.mysql.cj.jdbc.Driver")
                    .mode("append")
                    .save()
                )
                
            except Exception as e:
                logging.error(f"ERROR - {str(e)}")

        finance_data = df.select("symbol","timestamp","open","high","low","close","volume","market_cap")
        (
            finance_data
            .write
            .format("jdbc")
            .option("url","jdbc:mysql://host.docker.internal:3306/kafka_spark")
            .option("dbtable","finance")
            .option("user","root")
            .option("password","abdellah2004")
            .option("driver","com.mysql.cj.jdbc.Driver")
            .mode("append")
            .save()
        )
    
    except Exception as e:
        logging.error(f"ERROR - {str(e)}")

In [ ]:
(
    json_kafka
    .writeStream
    .foreachBatch(save_to_mysql)
    .outputMode("append")
    .trigger(processingTime="5 seconds")
    .option("checkpointLocation","checkpint-dir-kafka")
    .start()
    .awaitTermination()
)

________________

In [15]:
# First, read existing sectors from MySQL
existing_data = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:mysql://host.docker.internal:3306/kafka_spark") \
    .option("dbtable", "sector") \
    .option("user", "root") \
    .option("password", "abdellah2004") \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .load()

# Get list of existing sector names
existing_data = [row.sector_name for row in existing_data.collect()]

# Get new sectors from Kafka data
new_data = json_kafka.select("sector_name").distinct()

# Filter out sectors that already exist
data_to_insert = new_data.filter(~col("sector_name").isin(existing_data))

# Insert only new sectors
if data_to_insert.count() > 0:
    (
        data_to_insert.write
        .format("jdbc")
        .option("url", "jdbc:mysql://host.docker.internal:3306/kafka_spark") 
        .option("dbtable", "sector")
        .option("user", "root")
        .option("password", "abdellah2004")
        .option("driver", "com.mysql.cj.jdbc.Driver")
        .mode("append")
        .save()
    )
    print(f"Inserted {data_to_insert.count()} new sectors")

Inserted 3 new sectors


In [23]:
sectors = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:mysql://host.docker.internal:3306/kafka_spark") \
    .option("dbtable", "sector") \
    .option("user", "root") \
    .option("password", "abdellah2004") \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .load()

sectors.show()

+---------+--------------------+
|id_sector|         sector_name|
+---------+--------------------+
|        3|Communication Ser...|
|        1|   Consumer Cyclical|
|        2|          Technology|
+---------+--------------------+



In [24]:
countries = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:mysql://host.docker.internal:3306/kafka_spark") \
    .option("dbtable", "country") \
    .option("user", "root") \
    .option("password", "abdellah2004") \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .load()

countries.show()

+----------+-------------+
|id_country| country_name|
+----------+-------------+
|         1|United States|
+----------+-------------+



In [25]:
industries = spark.read \
        .format("jdbc") \
        .option("url","jdbc:mysql://host.docker.internal:3306/kafka_spark") \
        .option("dbtable","industry") \
        .option("user","root") \
        .option("password","abdellah2004") \
        .option("driver", "com.mysql.cj.jdbc.Driver") \
        .load()
industries.show()

+------+--------------------+
|id_ind|            name_ind|
+------+--------------------+
|     2|  Auto Manufacturers|
|     1|Consumer Electronics|
|     6|       Entertainment|
|     4|Internet Content ...|
|     7|     Internet Retail|
|     3|      Semiconductors|
|     5|Software - Infras...|
+------+--------------------+



In [26]:
json_kafka.show()

+------+-------------------+-------+------+--------+------+-------+------------+--------------------+--------------------+-------------+--------+-------------+-----+--------------------+
|symbol|          timestamp|   open|  high|     low| close| volume|company_name|         sector_name|            name_ind| country_name|currency|   market_cap|  zip|             website|
+------+-------------------+-------+------+--------+------+-------+------------+--------------------+--------------------+-------------+--------+-------------+-----+--------------------+
|  AAPL|2025-12-31 20:59:00|272.045|272.13|  271.75|271.84| 729227|       Apple|          Technology|Consumer Electronics|United States|     USD|4034508095488|95014|https://www.apple...|
| GOOGL|2025-12-31 20:59:00| 313.37|313.43|  312.96| 313.0| 336596|    Alphabet|Communication Ser...|Internet Content ...|United States|     USD|3791094022144|94043|     https://abc.xyz|
|  MSFT|2025-12-31 20:59:00| 483.77|483.88|  483.38|483.65| 35430

In [28]:
company = json_kafka.join(sectors,on="sector_name",how="left") \
    .join(countries,on="country_name",how="left") \
    .join(industries,on="name_ind",how="left") \
    .select("symbol","company_name","website","id_sector","id_ind","id_country","currency","zip")
company.show()

+------+------------+--------------------+---------+------+----------+--------+-----+
|symbol|company_name|             website|id_sector|id_ind|id_country|currency|  zip|
+------+------------+--------------------+---------+------+----------+--------+-----+
|  AMZN|      Amazon|https://www.amazo...|        1|     7|         1|     USD|98109|
|  TSLA|       Tesla|https://www.tesla...|        1|     2|         1|     USD|78725|
|  AAPL|       Apple|https://www.apple...|        2|     1|         1|     USD|95014|
| GOOGL|    Alphabet|     https://abc.xyz|        3|     4|         1|     USD|94043|
|  MSFT|   Microsoft|https://www.micro...|        2|     5|         1|     USD|98052|
|  META|        Meta|https://investor....|        3|     4|         1|     USD|94025|
|  NVDA|      NVIDIA|https://www.nvidi...|        2|     3|         1|     USD|95051|
|  NFLX|     Netflix|https://www.netfl...|        3|     6|         1|     USD|95032|
+------+------------+--------------------+---------+--

In [29]:
(
    company.write
    .format("jdbc")
    .option("url", "jdbc:mysql://host.docker.internal:3306/kafka_spark") 
    .option("dbtable", "company")
    .option("user", "root")
    .option("password", "abdellah2004")
    .option("driver", "com.mysql.cj.jdbc.Driver")
    .mode("append")
    .save()
)

In [32]:
finance_data = json_kafka.select("symbol","timestamp","open","high","low","close","volume","market_cap")
finance_data.show()

+------+-------------------+-------+------+--------+------+-------+-------------+
|symbol|          timestamp|   open|  high|     low| close| volume|   market_cap|
+------+-------------------+-------+------+--------+------+-------+-------------+
|  AAPL|2025-12-31 20:59:00|272.045|272.13|  271.75|271.84| 729227|4034508095488|
| GOOGL|2025-12-31 20:59:00| 313.37|313.43|  312.96| 313.0| 336596|3791094022144|
|  MSFT|2025-12-31 20:59:00| 483.77|483.88|  483.38|483.65| 354308|3594827857920|
|  AMZN|2025-12-31 20:59:00|  230.9|230.95|   230.7|230.84| 461738|2467515596800|
|  TSLA|2025-12-31 20:59:00| 449.81|449.83|  449.44|449.55| 614642|1495687364608|
|  META|2025-12-31 20:59:00| 660.29|660.43|659.7775| 659.9| 140732|1663775145984|
|  NVDA|2025-12-31 20:59:00| 186.69|186.72|  186.49|186.49|1908595|4540715761664|
|  NFLX|2025-12-31 20:59:00| 93.765| 93.78|   93.72| 93.76| 366068| 397291454464|
+------+-------------------+-------+------+--------+------+-------+-------------+



In [34]:
(
    finance_data
    .write
    .format("jdbc")
    .option("url","jdbc:mysql://host.docker.internal:3306/kafka_spark")
    .option("dbtable","finance")
    .option("user","root")
    .option("password","abdellah2004")
    .option("driver","com.mysql.cj.jdbc.Driver")
    .mode("append")
    .save()
)